# Servidor

En este documentos veremos como es la implementación de un servidor de parametros hecho con python + sockets + pickel + json, también analizaremos si el entrenamiento es efectivo con mnist y revisaremos el tiempo que se demora con diferentes cantidad de workers.

Importamos algunas herramientos que se comparten entre servidor y cliente.

In [1]:
from common import *

/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h2>Maquina de estados</h2>
Declaramos nuestra maquina de estados, esta nos ayudara a sincronizar los diversos trabajadores en etapas como:
<ul>
<li>
<b>handshake:</b> En esta etapa, el servidor espera la conección de los diversos trabajadores, enviandoles la información basica para comenzar a repartir el trabajo.
</li>
<li>
<b>recolection:</b> en esta etapa, los trabajadores confirmar con el servidor que todo fue bién y comienza a preparar el dataset, si la confirmación es negativa, el trabajador se saca de la lsita de trabajadores y se comienza con la siguiente etapa sin el.
</li>
<li>
<b>trainBacth:</b> En esta etapa, el servidor envia los pesos a los trabajadores y ellos comienzan a entrenar, cada trabajador se reparte una parte del bacth.
</li>
<li>
<b>validation:</b> Estapa opcional, que solo se utiliza cuando se quiere medir la capacidad de la red neuronal, cada trabajador ejecuta un test de validación sobre el mismo bacth que entreno y parte del test de validación, envia los resultados al servidor.
</li>
<li>
<b>end:</b> Al final, los trabajadores le envian los gradientes ya entrenados, el servidor hace un promedio y entrena los pesos, si ya se analizo todo el dataset, se pasa de epoca, si ya no hay epocas para entrenar, se le dice a los trabajadores que el trabajo ya fue hecho y que terminen de trabajar.
</li>
</ul>

In [18]:
from socket import socket
from typing import Dict, Any
import time
import pickle
import json
import os
from tqdm import tqdm

class State:
    
    def __init__(self, params: Dict[str, Any]):
        self.data = params
    
    def do(self, sock: socket):
        self.data["status"] = "ok"
    
    def next(self):
        if self.data.get("status") == "error":
            return None
        return HandShake(self.data)

class HandShake(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            workers = self.data["workers"]
            workersList = []
            self.data["startTime"] = time.time_ns()
            params = {
                "sequential": self.data["sequential"].export(), 
                "error": self.data["error"],
                "devError": self.data["devError"],
                "dsName": self.data["dsName"],
                "split": self.data["split"],
                "shardPosition": 0,
                "batchSize": self.data["batchSize"],
                "labels": self.data["labels"],
                "seed": self.data["seed"],
                "token": os.getenv("token"),
                "workers": self.data["workers"],
                "datasetPorcent": self.data["datasetPorcent"],
                "shape": [self.data["shape"][0], self.data["shape"][1]],
                "labelsNumber": self.data["labelsNumber"],
                "verbose": self.data["verbose"],
                "test": self.data["test"]
            }
            (self.data["ds"], self.data["dsSize"]) = downloadDataset(
                self.data["dsName"], 
                "train"
            )
            if self.data["test"]:
                (_, self.data["dsTestSize"]) = downloadDataset(
                    self.data["dsName"], 
                    "test"
                )
            overheadStart = time.time_ns()
            while workers != 0:
                conn, address = sock.accept()
                print(f"se ha conectado {address}")
                workers-=1
                workersList.append(conn)
                params["shardPosition"] = workers
                json_data = json.dumps(params).encode("utf-8")
                conn.sendall(len(json_data).to_bytes(8, 'big'))
                conn.sendall(json_data)
            self.data["tqdm"] = tqdm(total=self.data["epochs"], desc="Epochs")
            self.data["overheadTime"] = time.time_ns() - overheadStart
            self.data["workersList"] = workersList
        except Exception as e:
            print("HandShake error:", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        return Recolection(self.data)
    
class Recolection(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            workers = len(self.data["workersList"])
            trueList = []
            self.data["a"] = 0
            for i in range(workers):
                worker_socket = self.data["workersList"][i]
                ans = recvall(worker_socket, 3).decode("utf-8")
                cod, workerNumber = ans.split("-")
                if(cod == "y"):
                    trueList.append(self.data["workersList"][int(workerNumber)])
                    continue
                self.data["workersList"][i].close()
            if len(trueList) == 0:
                raise Exception("Todos los workers fallaron la recolección")
            self.data["wbw"] = self.data["batch"] // len(trueList)
            self.data["workersList"] = trueList
        except Exception as e:
            print("Recolection error:", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        return TrainBatch(self.data)

class TrainBatch(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            a = self.data["a"]
            if a >= math.ceil((self.data["dsSize"]*self.data["datasetPorcent"])/self.data["batch"]):
                self.data["epochs"] -= 1
                self.data["a"] = 0
                if "tqdm" in self.data:
                    self.data["tqdm"].update(1)
            wb_data = pickle.dumps((self.data["w"], self.data["b"]))
            activeList = []
            for conn in self.data["workersList"]:
                try:
                    conn.sendall(len(wb_data).to_bytes(8, 'big'))
                    conn.sendall(wb_data)
                    activeList.append(conn)
                except Exception as ex:
                    print("A worker was disconnected while sending weights:", ex)
            
            self.data["workersList"] = activeList
            if not activeList:
                raise Exception("No active workers remaining to train batch.")
                
        except Exception as e:
            print("TrainBatch error:", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        if self.data["verbose"]:
            return Validate(self.data)
        return End(self.data)

class Validate(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            accuracies = 0
            test_accuracies = 0
            all_losses = []
            activeList = []
            for conn in self.data["workersList"]:
                try:
                    length_prefix = recvall(conn, 8)
                    message_length = int.from_bytes(length_prefix, 'big')
                    data = recvall(conn, message_length)
                    acc, test_acc, batch_losses = pickle.loads(data)
                    accuracies += acc
                    test_accuracies += test_acc
                    all_losses.extend(batch_losses)
                    activeList.append(conn)
                except Exception as ex:
                    print(f"Un worker falló al enviar validación: {ex}")
            self.data["workersList"] = activeList
            if not activeList:
                raise Exception("No quedan trabajadores activos para compilar el history.")
            avg_acc = accuracies / self.data["batch"]
            avg_test_acc = 0
            if self.data["test"]:
                avg_test_acc = test_accuracies / self.data["dsTestSize"]
            avg_loss = float(np.mean(all_losses)) if all_losses else 0.0
            self.data["history"].append({
                "epoch": self.data["epochs"],
                "batch": self.data["a"],
                "loss_avg": avg_loss,
                "accuracy": avg_acc,
                "test_accuracy": avg_test_acc,
                "time": 0,
                "overheadTime": self.data["overheadTime"] / 1000,
                "harvestTime": 0,
                "sumTime": 0
            })
        except Exception as e:
            print("Validate error:", e)
            self.data["status"] = "error"
    
    def next(self):
        if self.data.get("status") == "error":
            return None
        return End(self.data)

class End(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            sequential = self.data["sequential"]
            learningRate = self.data["learningRate"]
            batch = self.data["batch"]
            harvestStart = time.time_ns()
            activeList = []
            for conn in self.data["workersList"]:
                try:
                    ans = recvall(conn, 3).decode("utf-8")
                    cod, workerNumber = ans.split("-")
                    if cod == "y":
                        length_prefix = recvall(conn, 8)
                        message_length = int.from_bytes(length_prefix, 'big')
                        data = recvall(conn, message_length)
                        w_grad_batch, b_grad_batch = pickle.loads(data)
                        for i in range(len(sequential) - 1):
                            self.data["w"][i] -= learningRate * (w_grad_batch[i] / batch)
                            self.data["b"][i+1] -= learningRate * (b_grad_batch[i+1] / batch)
                        activeList.append(conn)
                except Exception as ex:
                    print("A worker failed during gradient harvesting:", ex)
            self.data["workersList"] = activeList
            if not activeList:
                raise Exception("No active workers could finalize the batch.")
            self.data["harvestTime"] = time.time_ns() - harvestStart
            self.data["history"][-1]["harvestTime"] = (time.time_ns() - harvestStart)/1000
            self.data["history"][-1]["time"] = (time.time_ns() - self.data["startTime"]) / 1000
        except Exception as e:
            print("End error:", e)
            self.data["status"] = "error"
            
    def next(self):
        if self.data.get("status") == "error":
            return None
        self.data["a"] += 1
        if self.data["epochs"] > 0:
            for conn in self.data["workersList"]:
                try:
                    conn.sendall(b"y")
                except:
                    pass
            return TrainBatch(self.data)
        if "tqdm" in self.data:
            try:
                self.data["tqdm"].close()
            except: 
                pass
        for conn in self.data["workersList"]:
            try:
                conn.sendall(b"n")
                conn.close() 
            except:
                pass
        return None

declaramos el metodo fir, el cual, no ayudara a cambiar de estado y darnos el resultado de la maquina de estados.

In [19]:
from typing import Callable, Tuple, List

def fit(
    dataset: str,
    epochs: int,
    learningRate: float,
    sequential,
    error: Callable,
    devError: Callable,
    InitB: float,
    batch: int = 1,
    workers: int = 1,
    datasetPorcent: int = 1,
    shape: Tuple[int, int] = (224, 224),
    labels: int = 10000,
    verbose:bool = False,
    test: bool = False,
    labelsName: List[str] = ["image", "label"]
):
    import socket
    os.environ.pop("OMP_NUM_THREADS", None) 
    os.environ.pop("OPENBLAS_NUM_THREADS", None) 
    os.environ.pop("MKL_NUM_THREADS", None)
    os.environ.pop("VECLIB_MAXIMUM_THREADS", None)
    os.environ.pop("NUMEXPR_NUM_THREADS", None)
    w = []
    b = []
    history = []
    seed = int(os.getenv("seed", 1))
    for i in range(len(sequential)):
        if i < len(sequential) - 1:
            s = (sequential[i].neurons, sequential[i+1].neurons)
            limit = np.sqrt(6 / (s[0] + s[1]))
            w.append(np.random.uniform(-limit, limit, size=s))
        b.append(np.full((sequential[i].neurons,), InitB))
    estado_actual = State({
        "sequential": sequential, 
        "error": error.__name__,
        "devError": devError.__name__,
        "dsName": dataset,
        "split": "train",
        "shard": 0,
        "datasetPorcent": datasetPorcent,
        "workers":workers,
        "shardPosition": 0,
        "batchSize": batch,
        "batch": batch,
        "seed": seed,
        "labels": labelsName,
        "w": w,
        "b": b,
        "epochs": epochs,
        "learningRate": learningRate,
        "verbose": verbose,
        "test": test,
        "history": history,
        "shape": shape,
        "labelsNumber": labels
    })
    HOST = "127.0.0.1"
    PORT = 65432
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORT))
        s.listen()
        while estado_actual is not None:
            estado_actual.do(s)
            estado_actual = estado_actual.next()
    return (Model(sequential, w, b), history)

declaramos la red para predecir cifar 10

In [20]:
os.environ["token"] = ""
(modelSerial, historyComplete) = fit(
    dataset="uoft-cs/cifar10",
    epochs=1,
    learningRate=0.1,
    sequential=Sequential(
        Layer(1024, relu, devRelu, "input"),
        Layer(512, relu, devRelu, "hidden"),
        Layer(128, relu, devRelu, "hidden"),
        Layer(10, softmax, devSoftmax, "output")
    ),
    error=lostEntropy,
    devError=devLostEntropy,
    InitB=0.15,
    batch=16384,
    verbose=True,
    datasetPorcent=1,
    shape=(32, 32),
    labels=10,
    test=True,
    labelsName=["img", "label"]
)

se ha conectado ('127.0.0.1', 53720)


Epochs: 100%|██████████| 1/1 [03:13<00:00, 193.00s/it]


In [16]:
import pandas as pd
import altair as alt
from sklearn.metrics import confusion_matrix
import io
import base64
from PIL import Image

def plot_loss(history):
    df_history = pd.DataFrame(history)
    df_epoch = df_history.groupby('epoch')['loss_avg'].mean().reset_index()
    line = alt.Chart(df_epoch).mark_line(color='#2196F3', strokeWidth=3).encode(
        x=alt.X('epoch:Q', title='Época', axis=alt.Axis(tickMinStep=1)),
        y=alt.Y('loss_avg:Q', title='Pérdida (loss_avg)', scale=alt.Scale(type='log')), # Escala logarítmica suele ser mejor para loss_avg
        tooltip=['epoch', 'loss_avg']
    )
    points = line.mark_point(size=60, filled=True).encode(
        opacity=alt.value(1)
    )
    return (line + points).properties(
        title='Progreso del Entrenamiento: loss_avg vs Epoch',
        width=600,
        height=300
    ).interactive()

def plot_confusion_matrix(m, x, y, label_names):
    y_pred = np.zeros_like(y)
    for i in range(len(x)):
        y_pred[i] = np.argmax(m.fordward(x[i]))
    cm = confusion_matrix(y, y_pred)
    data = []
    for i in range(len(cm)):
        for j in range(len(cm)):
            data.append({
                'Actual': str(label_names[i]),
                'Predicho': str(label_names[j]),
                'Cantidad': int(cm[i, j])
            })
    df_cm = pd.DataFrame(data)
    threshold = float(cm.max() / 2)
    base = alt.Chart(df_cm).encode(
        x=alt.X('Predicho:O', title='Clase Predicha', sort=list(label_names.values())),
        y=alt.Y('Actual:O', title='Clase Real', sort=list(label_names.values()))
    )
    heatmap = base.mark_rect().encode(
        color=alt.Color('Cantidad:Q', scale=alt.Scale(scheme='blues'), title='Frecuencia'),
        tooltip=['Actual', 'Predicho', 'Cantidad']
    )
    text = base.mark_text(baseline='middle').encode(
        text='Cantidad:Q',
        color=alt.condition(
            f"datum.Cantidad > {threshold}",
            alt.value('white'),
            alt.value('black')
        )
    )
    return (heatmap + text).properties(
        title='Matriz de Confusión (CIFAR-10)',
        width=500,
        height=500,
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14
    )

def plot_accuracy_only(history):
    df_history = pd.DataFrame(history)
    df_epoch = df_history.groupby('epoch')['accuracy'].mean().reset_index()
    line = alt.Chart(df_epoch).mark_line(color='#2196F3', strokeWidth=3).encode(
        x=alt.X('epoch:Q', title='Época', axis=alt.Axis(tickMinStep=1)),
        y=alt.Y('accuracy:Q', title='Pérdida (accuracy)', scale=alt.Scale(type='log')), # Escala logarítmica suele ser mejor para accuracy
        tooltip=['epoch', 'accuracy']
    )
    points = line.mark_point(size=60, filled=True).encode(
        opacity=alt.value(1)
    )
    return (line + points).properties(
        title='Progreso del Entrenamiento: accuracy vs Epoch',
        width=600,
        height=300
    ).interactive()

def plot_time(history):
    df_history = pd.DataFrame(history)
    df_history = df_history.rename(columns={'time': 'totalTime'})
    
    df_epoch = df_history.groupby('epoch').agg({
        'totalTime': 'mean',
        'overheadTime': 'mean',
        'harvestTime': 'mean',
        'sumTime': 'mean'
    }).reset_index()
    df_melted = df_epoch.melt(
        id_vars=['epoch'], 
        value_vars=['totalTime', 'overheadTime', 'harvestTime', 'sumTime'],
        var_name='Metrica', 
        value_name='ms'
    )
    nearest = alt.selection_point(nearest=True, on='mouseover', 
                                  fields=['epoch'], empty=False)
    lines = alt.Chart(df_melted).mark_line(strokeWidth=3).encode(
        x=alt.X('epoch:Q', title='Época'),
        y=alt.Y('ms:Q', title='Tiempo (ms)', scale=alt.Scale(type='symlog')), 
        color=alt.Color('Metrica:N', scale=alt.Scale(
            domain=['totalTime', 'overheadTime', 'harvestTime', 'sumTime'],
            range=['#2196F3', '#FF5722', '#000000', '#00913F']
        ))
    )
    selectors = alt.Chart(df_epoch).mark_rule(color='gray', strokeWidth=1).encode(
        x='epoch:Q',
        opacity=alt.condition(nearest, alt.value(0.3), alt.value(0)),
        tooltip=[
            alt.Tooltip('epoch:Q', title='Época'),
            alt.Tooltip('totalTime:Q', title='Tiempo Total (ms)', format='.2f'),
            alt.Tooltip('overheadTime:Q', title='Overhead (ms)', format='.2f'),
            alt.Tooltip('harvestTime:Q', title='Harvest (ms)', format='.2f'),
            alt.Tooltip('sumTime:Q', title='Sum (ms)', format='.2f'),
        ]
    ).add_params(nearest)
    points = lines.mark_point(size=80, filled=True).encode(
        opacity=alt.condition(nearest, alt.value(1), alt.value(0))
    )
    return (lines + selectors + points).properties(
        title='Análisis de Rendimiento',
        width=600,
        height=300,
    ).interactive()

def plot_wall_of_shame(m, x_test, y_test, label_names, num_errors=15):
    errors_found = []
    # y_test viene en one-hot, sacamos la clase categórica real
    y_test_categorical = np.argmax(y_test, axis=1)
    
    for i in range(len(x_test)):
        if len(errors_found) >= num_errors:
            break
        output = m.fordward(x_test[i])
        pred_idx = np.argmax(output)
        actual_idx = y_test_categorical[i]
        
        if pred_idx != actual_idx:
            # Reestructuramos la imagen a (28, 28) escalar usando los valores en x_test
            # Se asumen imágenes en rango [0, 1] que se escalan a 255.
            img_array = (x_test[i].reshape((28, 28)) * 255).astype(np.uint8)
            img_pil = Image.fromarray(img_array, mode="L") # Mode="L" para un solo canal (gris)
            
            buffered = io.BytesIO()
            img_pil.save(buffered, format="PNG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            
            actual_name = label_names.get(actual_idx, str(actual_idx))
            pred_name = label_names.get(pred_idx, str(pred_idx))
            
            errors_found.append({
                "image": f"data:image/png;base64,{img_b64}",
                "Actual": actual_name,
                "Predicho": pred_name,
                "Leyenda": f"Real: {actual_name} | Pred: {pred_name}",
                "Confianza": float(np.max(softmax(output))), # Retornamos la confianza con Softmax
                "row": len(errors_found) // 5,
                "col": len(errors_found) % 5
            })
            
    if not errors_found:
        print("No se encontraron errores en la muestra.")
        return alt.Chart(pd.DataFrame({'A':[]})).mark_point()
        
    df_errors = pd.DataFrame(errors_found)
    base = alt.Chart(df_errors).encode(
        x=alt.X('col:O', axis=alt.Axis(labels=False, ticks=False, title=None)),
        y=alt.Y('row:O', axis=alt.Axis(labels=False, ticks=False, title=None))
    )
    image_layer = base.mark_image(width=70, height=70).encode(
        url='image',
        tooltip=['Actual', 'Predicho', alt.Tooltip('Confianza:Q', format='.2%')]
    ).properties(
         width=600,
         height=400,
    )
    text_layer = base.mark_text(
        dy=55,
        fontSize=10,
        fontWeight='bold'
    ).encode(
        text='Leyenda:N',
        color=alt.value('#D32F2F')
    )
    return (image_layer + text_layer).properties(
        title="Wall of Shame: Errores del Modelo (MNIST)",
        padding=30
    ).configure_view(strokeWidth=0)
    
def plot_accuracy(history):
    df_history = pd.DataFrame(history)
    df_epoch = df_history.groupby('epoch').agg({
        'accuracy': 'mean',
        'test_accuracy': 'mean'
    }).reset_index()
    df_melted = df_epoch.melt(
        id_vars=['epoch'], 
        value_vars=['accuracy', 'test_accuracy'],
        var_name='Dataset', 
        value_name='Acc'
    )
    base = alt.Chart(df_melted).encode(
        x=alt.X('epoch:Q', title='Época', axis=alt.Axis(tickMinStep=1)),
        y=alt.Y('Acc:Q', title='Exactitud (Accuracy)', scale=alt.Scale(domain=[0, 1])),
        color=alt.Color('Dataset:N', scale=alt.Scale(
            domain=['accuracy', 'test_accuracy'],
            range=['#2196F3', '#FF9800']
        ), title='Leyenda')
    )
    lines = base.mark_line(strokeWidth=3)
    points = base.mark_point(size=60, filled=True)
    # selección más cercana para mostrar tooltips con ambos valores (train/test)
    nearest = alt.selection_point(nearest=True, on='mouseover', fields=['epoch'], empty='none')
    tooltip_rule = alt.Chart(df_epoch).mark_rule(color='gray').encode(
        x='epoch:Q',
        opacity=alt.condition(nearest, alt.value(0.3), alt.value(0)),
        tooltip=[
            alt.Tooltip('epoch:Q', title='Época'),
            alt.Tooltip('accuracy:Q', title='Train Acc', format='.2%'),
            alt.Tooltip('test_accuracy:Q', title='Test Acc', format='.2%')
        ]
    ).add_selection(nearest)
    return (lines + points + tooltip_rule).properties(
        title='Evolución del Accuracy: Entrenamiento vs. Prueba',
        width=600,
        height=300,
    ).interactive()

validamos como fue el entrenamiento:

In [21]:
# import os
# os.environ["token"] = os.getenv("HF_TOKEN", "")

# 1. Descargamos el dataset usando el nuevo formato
(dsTest, sizeTest) = downloadDataset("uoft-cs/cifar10", "test")

# 2. Iteramos para traer un lote grande de prueba (como si tuvieramos 1 solo worker para nosotros)
# Pasamos: split=(1,0) para agarrar todo el subset
# workerNumber=1 
# batchSize y classNumber adaptados
x, y = next(getBatch(
    dsTest, 
    batchSize=10000, 
    labels=("img", "label"), 
    size=sizeTest, 
    workerNumber=1, 
    split=(1, 0), 
    shape=(32, 32),
    label="test",
    classNumber=10
))

labelToMeaning = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}
plot_confusion_matrix(modelSerial, x, np.argmax(y, axis=1), labelToMeaning).show()

plot_accuracy(historyComplete).show()
plot_time(historyComplete).show()
plot_wall_of_shame(modelSerial, x, y, labelToMeaning).show()

alt.LayerChart(...)

/tmp/ipykernel_42660/158659127.py:224: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(nearest)


alt.LayerChart(...)

alt.LayerChart(...)

ValueError: cannot reshape array of size 1024 into shape (28,28)

ahora, analicemos el tiempo con diferentes trabajadores con una dataset mas interezante, imaginet

In [ ]:
times = []
maxI = 5
for worker in range(1, 4):
    for _ in range(maxI):
        start = time.time_ns()
        (modelImagenet, historyImagenet) = fit(
            dataset="ylecun/mnist",
            epochs=1,
            learningRate=0.1,
            sequential=Sequential(
                Layer(50176, relu, devRelu, "input"),
                Layer(1024, relu, devRelu, "hidden"),
                Layer(512, relu, devRelu, "hidden"),
                Layer(128, relu, devRelu, "hidden"),
                Layer(1000, softmax, devSoftmax, "output")
            ),
            error=lostEntropy,
            devError=devLostEntropy,
            InitB=0.15,
            batch=2098,
            verbose=False,
            datasetPorcent=0.05,
            shape=(224, 224),
            labels=1000,
            test=False
        )
        end = time.time_ns()
        times.append({
            "workers": worker,
            "time":(end-start)/1000,
        })

se ha conectado ('127.0.0.1', 59310)


Epochs: 100%|██████████| 2/2 [04:07<00:00, 123.68s/it]


In [19]:
plot_accuracy(historyImagenet).show()
plot_time(historyImagenet).show()

/tmp/ipykernel_30501/158659127.py:224: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(nearest)


alt.LayerChart(...)

alt.LayerChart(...)